# WaveletSpaceNet — sparse context + a wavelet image pyramid → mesh-plane + camera pose

This notebook clones the repo, installs it, downloads the **DIODE** validation split,
runs the tests, visualises a generated **fly-through**, trains on the Colab GPU and runs
inference (a grayscale frame + optional sparse context → a **mesh-plane** + **camera pose**).

> Runtime → *Change runtime type* → **GPU** before running the training cell.

## 1 · Clone the repository

In [ ]:
import os
if not os.path.exists('WaveletSpaceNet'):
    !git clone https://github.com/OlegJakushkin/WaveletSpaceNet.git
%cd WaveletSpaceNet
!git pull --ff-only || true

## 2 · Install dependencies

In [ ]:
!pip -q install -r requirements.txt

## 3 · Download DIODE (validation: 325 indoor + 446 outdoor)

~ a few GB; skipped if already present.  The tests and training fall back to a synthetic
scene generator if you skip this cell.

In [ ]:
import os, glob
os.makedirs('data/diode', exist_ok=True)
if not glob.glob('data/diode/**/*_depth.npy', recursive=True):
    !wget -q http://diode-dataset.s3.amazonaws.com/val.tar.gz -O val.tar.gz
    !tar xf val.tar.gz -C data/diode && rm -f val.tar.gz
print('DIODE depth views:', len(glob.glob('data/diode/**/*_depth.npy', recursive=True)))

## 4 · Tests — a single fly-through generates + trains locally

In [ ]:
!python -m pytest tests/ -q

## 5 · Visualise a generated fly-through

A random smooth curve → noised grayscale renders + ground-truth depth, plus the sparse
noised context cloud ('the points gathered before').

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from waveletspace import diode as D, geometry as G
root = D.find_diode_root()
rng = np.random.default_rng(0)
scene = D.scene_from_view(D.list_views(root)[0]) if root else D.synthetic_scene(rng)
Rs, ts = G.flythrough(scene.centroid, scene.extent, rng, n_frames=6)
fig, ax = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    g, d, m, _ = D.noised_render(scene, Rs[i], ts[i], 256, 64, rng)
    ax[0, i].imshow(g, cmap='gray'); ax[0, i].set_title(f'frame {i}'); ax[0, i].axis('off')
    ax[1, i].imshow(np.where(d > 0, d, np.nan), cmap='turbo'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()
ctx = D.sample_context(scene, 512, rng)
print('sparse noised context:', ctx.shape, '| scene name:', scene.name)

## 6 · Train on the Colab GPU

The default model uses the full `1024,512,256,128,64,32` wavelet pyramid; that is heavy,
so for a Colab session we cap the top level at 512.  Increase `--levels` / `--epochs` for a
real run.

In [ ]:
!python train.py --epochs 6 --batch 8 --img-hw 256 --plane-res 64 \
    --levels 512,256,128,64,32 --d 256 --M 256 --L 6 --out waveletspace

## 7 · Training curve + render panel

In [ ]:
import json, glob
from IPython.display import Image as IPImage, display
hist = json.load(open('renders/ws_train_hist.json'))
print('val chamfer(m) per epoch:', [round(h['val']['chamfer'], 3) for h in hist])
print('val logdepthL1 per epoch:', [round(h['val']['logdepthL1'], 3) for h in hist])
panels = sorted(glob.glob('renders/ws_val_ep*.png'))
if panels: display(IPImage(panels[-1]))

## 8 · Inference → mesh-plane + camera pose

`--demo` runs one generated fly-through frame; or pass `--image frame.png [--context ctx.npy]`.

In [ ]:
!python infer.py --demo --ckpt assets/waveletspace.pt --out out/plane.obj
print('\nwrote out/plane.obj (download it / open in a mesh viewer)')